# HEP Multiagent Demo

This notebook demonstrates how to use the multi-agent framework for scientific research queries.

## Prerequisites
- Python 3.10+
- LaTeX distribution for PDF reports ([BasicTeX](https://www.tug.org/mactex/morepackages.html))
- Environment variables configured (see below)

In [1]:
# Development mode
# !pip install -e ".[dev]"

# For production:
# !pip install -q --force-reinstall git+https://github.com/HEP-KE/HEP-multiagent.git


In [2]:
# !pip install -q --force-reinstall git+https://github.com/HEP-KE/mcp-ke.git
#install last since it needs mcp<1.23.0 but 1.26.0 got installed
# !pip install -q --force-reinstall git+https://github.com/HEP-KE/kb-mcp.git

### Set up for KB MCP

Set up paths and choose which papers to download.

In [3]:
import os
import subprocess
import sys
from pathlib import Path

# Paths (stores data in current directory)
DATA_DIR = Path.cwd() / "data"
DB_PATH = DATA_DIR / "kb.db"
PAPERS_DIR = DATA_DIR / "papers"

# Papers to download (arXiv ID, title)
PAPERS = [
    ("1807.06209", "Planck 2018 cosmological parameters"),
    ("2007.08991", "eBOSS cosmological results"),
    ("1502.01589", "Planck 2015 cosmological results"),
]

print(f"Database: {DB_PATH}")
print(f"Papers: {PAPERS_DIR}")

Database: /data/a/cpac/nramachandra/Projects/AmSC/HEP-multiagent/data/kb.db
Papers: /data/a/cpac/nramachandra/Projects/AmSC/HEP-multiagent/data/papers


Initialize an empty SQLite database with the kb-mcp schema.

In [4]:
# from kb_mcp.kb.db_models import Base
# from sqlalchemy import create_engine

# # Create directories
# DB_PATH.parent.mkdir(parents=True, exist_ok=True)
# PAPERS_DIR.mkdir(parents=True, exist_ok=True)

# # Create database
# engine = create_engine(f"sqlite:///{DB_PATH}")
# Base.metadata.create_all(engine)

# print(f"Created database: {DB_PATH}")

Fetch PDFs from arXiv and extract text.

In [5]:
# from hep_multiagent.features.arxiv_fetch import download_full_text

# for arxiv_id, title in PAPERS:
#     txt_path = PAPERS_DIR / f"{arxiv_id}.txt"
#     if txt_path.exists():
#         print(f"{arxiv_id} - already downloaded")
#     else:
#         print(f"[downloading] {arxiv_id}: {title}")
#         download_full_text(arxiv_id, str(PAPERS_DIR))

# print(f"\nDownloaded {len(list(PAPERS_DIR.glob('*.txt')))} papers")

Ingest the downloaded papers into kb-mcp.

In [6]:
# # NOTE: Embeddings required for kb_search to work with SQLite.
# # Using --no-embed causes kb_search to crash (KeyError: 'total_results')
# # because SQLite doesn't support full-text search.

# os.environ["SQLITE_DB_PATH"] = str(DB_PATH)

# for arxiv_id, _ in PAPERS:
#     txt_path = PAPERS_DIR / f"{arxiv_id}.txt"
#     if txt_path.exists():
#         print(f"[ingesting] {arxiv_id}")
        
#         subprocess.run(
#             [sys.executable, "-m", "kb_mcp.kb.cli", "ingest", str(txt_path),
#              "--source-id", "arxiv", "--no-summary", "--batch"],
#             capture_output=True
#         )

# print("\nDone! Checking database...")
# result = subprocess.run([sys.executable, "-m", "kb_mcp.kb.cli", "stats"], capture_output=True, text=True)
# print(result.stdout)

### Set up HEP Multiagent

Set `ARGO_USER` in `.env` or pass env vars directly in server config.

In [7]:
import os
from datetime import datetime
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from hep_multiagent import Agent

load_dotenv(".env")

llm = ChatOpenAI(
    # model="claudesonnet4",
    model="claudeopus46",
    base_url="https://apps-dev.inside.anl.gov/argoapi/v1",
    api_key=os.environ.get("ARGO_USER", "")
)

## Initialize Agent

Create an agent with an LLM and MCP servers. Servers are auto-installed from URL at init.

In [8]:
from hep_multiagent import Agent

agent = await Agent(
    llm=llm,
    mcp_servers=[
    # {
    #     "url": "https://github.com/HEP-KE/kb-mcp.git",
    #     "name": "kb-server-stdio",
    #     "env": {"SQLITE_DB_PATH": str(DB_PATH)},
    # },
    {
        "url": "/data/a/cpac/nramachandra/Projects/AmSC/mcp-ke"
    },
    ],
    approval=False,
)

for tool in agent.tools:
    print(f"  - {tool.name}")

  - list_agent_files
  - load_array
  - load_dict
  - save_array
  - save_dict
  - compute_all_models
  - compute_power_spectrum
  - compute_suppression_ratios
  - get_lcdm_params
  - get_nu_mass_params
  - get_wcdm_params
  - create_theory_k_grid
  - load_observational_data
  - analyze_mcmc_samples
  - compute_best_fit_power_spectrum
  - create_mcmc_corner_plot
  - create_mcmc_trace_plot
  - run_mcmc_cosmology
  - plot_power_spectra
  - plot_suppression_ratios
  - arxiv_agent
  - download_arxiv_paper
  - download_full_arxiv_paper
  - list_files
  - read_text_file
  - search_arxiv
  - power_spectrum_agent


In [9]:
agent.print_tools()

- list_agent_files
- load_array
- load_dict
- save_array
- save_dict
- compute_all_models
- compute_power_spectrum
- compute_suppression_ratios
- get_lcdm_params
- get_nu_mass_params
- get_wcdm_params
- create_theory_k_grid
- load_observational_data
- analyze_mcmc_samples
- compute_best_fit_power_spectrum
- create_mcmc_corner_plot
- create_mcmc_trace_plot
- run_mcmc_cosmology
- plot_power_spectra
- plot_suppression_ratios
- arxiv_agent
- download_arxiv_paper
- download_full_arxiv_paper
- list_files
- read_text_file
- search_arxiv
- power_spectrum_agent


Simple test question

In [10]:
# timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
# OUTPUT_DIR = f"./output_hepke_arxiv_demo_{timestamp}"

# result = await agent.run(
#     query = "Search arxiv for 5 papers on dark matter detection, Briefly provide information on the most recent method. then create a bar chart showing the publication year distribution of the papers found.",
#     output_dir=OUTPUT_DIR,
# )

result = await agent.run(
    query = """
Using the observational data from eBOSS DR14 Lyman-alpha forest, 
compare the linear P(k) values for LCDM, LCDM with massive neutrinos (Emv=0.10 eV), and dark 
energy model with equation of state parameter w0=-0.9.

Create visualizations showing:
1. The power spectra comparison with observational data
2. The suppression ratios relative to LCDM

Comment on how close the P(k) values are and analyze the power spectrum suppression compared to LCDM.
""",
    output_dir="./output_kb_hep_mcp"
)

print(result)

In [11]:
mcmc_query = """Run everything via MCP tools in mcp-ke. 
Absolutely do not use your own code (no writing new python codes, existing tools should be used). 
Strictly no fake/synthetic/placeholder data. 
If something doesn't work, do not try to find a workaround or shortcuts that need writing realistic estimations. 

(1) First, load observational data from eBOSS (see: /data/a/cpac/nramachandra/Projects/AmSC/mcp-ke/data/DR14_pm3d_19kbins.txt).
(2) Then compare the P(k) with wCDM, ΛCDM + Massive Neutrinos and ΛCDM (use any set of parameters you need). 
Plot them all and also show the P(k) ratio with ΛCDM for the non-standard models. 
(3) Finally, run a full posterior analysis using MCMC. Do this for 4 parameters (sigma8, OmegaM, Σmν, N_species) of the 
ΛCDM + massive neutrinos model. Choose a Gaussian likelihood with errors from observations. 
Priors can be tight gaussian, centered about Planck-2018 constraints. 
(4) Show me the final posterior distribution plot from GetDist and the best-fit estimates.
(5) Provide a comprehensive final report. 
"""

In [12]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = f"./output_hepke_mcmc_demo_{timestamp}"
print(OUTPUT_DIR)

result = await agent.run(
    query = mcmc_query,
    output_dir=OUTPUT_DIR,
)

./output_hepke_mcmc_demo_20260209_001757
